In [9]:
import torch
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import DataLoader, TensorDataset 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)


In [10]:
class AutoencoderDenoisingCNN(nn.Module):
    def __init__(self, input_leight=64, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2),
            
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            
        )
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear((input_leight // 4) * 128, latent_dim)
        self.fc2 = nn.Linear(latent_dim, (input_leight // 4) * 128)
        
        self.decoder = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Upsample(scale_factor=2),

            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Upsample(scale_factor=2),

            nn.Conv1d(64, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),

            nn.Conv1d(32, 1, kernel_size=5, padding=2)
        )
    def forward(self, x): 
        x = x.permute(0, 2, 1)
        
        x = self.encoder(x)
        x = self.flatten(x)
        
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        
        x = x.view(x.size(0), 128, -1)
        x = self.decoder(x)
        x = x.permute(0, 2, 1)
        return x
    
            
            
        

In [11]:
class TDNNDenosier(nn.Module):
    def __init__(self, input_length=64, time_delays=(1,2,4,8)):
        super().__init__()
        self.time_delays = time_delays
        input_dim = len(time_delays) + 1
        
        self.net = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.BatchNorm1d(input_length),
        nn.Dropout(0.2),

        nn.Linear(128, 64),
        nn.ReLU(),
        nn.BatchNorm1d(input_length),
        nn.Dropout(0.2),

        nn.Linear(64, 32),
        nn.ReLU(),

        nn.Linear(32, 1)
        )

    def forward(self, x):
        # x: (batch, length, 1)
        batch, length, _ = x.shape

        features = [x]

        for d in self.time_delays:
            pad = torch.zeros(batch, d, 1, device=x.device)
            delayed = torch.cat([pad, x[:, :-d, :]], dim=1)
            features.append(delayed)

        x = torch.cat(features, dim=2)  # (batch, length, channels)

        x = self.net(x)
        return x

In [12]:
def train_denoising_model(model, X_train_noisy, X_train_clean,
                         X_val_noisy=None, X_val_clean=None,
                         epochs=50, batch_size=32, model_name="CNN"):

    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*60}")

    model = model.to(device)

    train_dataset = TensorDataset(
        torch.tensor(X_train_noisy, dtype=torch.float32),
        torch.tensor(X_train_clean, dtype=torch.float32)
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    if X_val_noisy is not None:
        val_dataset = TensorDataset(
            torch.tensor(X_val_noisy, dtype=torch.float32),
            torch.tensor(X_val_clean, dtype=torch.float32)
        )
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss = float("inf")
    patience, patience_counter = 10, 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        if X_val_noisy is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    pred = model(xb)
                    val_loss += criterion(pred, yb).item()

            val_loss /= len(val_loader)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_weights = model.state_dict()
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print("Early stopping triggered")
                model.load_state_dict(best_weights)
                break

        if epoch % 5 == 0:
            print(f"Epoch {epoch}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

    return model

In [13]:
def evaluate_denoiser(model, X_test_noisy, X_test_clean, model_name="CNN"):

    print(f"\n{'='*60}")
    print(f"EVALUATION: {model_name}")
    print(f"{'='*60}")

    model.eval()

    X_test_noisy_t = torch.tensor(X_test_noisy, dtype=torch.float32).to(device)

    with torch.no_grad():
        preds = model(X_test_noisy_t).cpu().numpy()

    mse_noisy = np.mean((X_test_noisy - X_test_clean) ** 2)
    mse_denoised = np.mean((preds - X_test_clean) ** 2)

    mae_noisy = np.mean(np.abs(X_test_noisy - X_test_clean))
    mae_denoised = np.mean(np.abs(preds - X_test_clean))

    snr_noise = 10 * np.log10(np.mean(X_test_clean ** 2) / mse_noisy)
    snr_denoised = 10 * np.log10(np.mean(X_test_clean ** 2) / mse_denoised)

    print(f"MSE (noisy): {mse_noisy:.6f}")
    print(f"MSE (denoised): {mse_denoised:.6f}")
    print(f"SNR improvement: {snr_denoised - snr_noise:.2f} dB")

    return preds